In [ ]:
from spectra_codec import SpectraCodec
import numpy as np
import matplotlib.pyplot as plt

# Comparing Message Length Scaling: Sequential-ASCII vs Hilbert Curve Encoding

We analyze how each encoding method scales with message length using the following default parameters, chosen to accommodate the constraints of 32-bit precision in mzML files and the practical limits of mass spectrometry data:

- `min_mz = 5` (minimum m/z value to avoid obscure low-mass regions)
- `mz_scale = 0.0001` (precision scaling factor)
- `min_intensity = 100` (minimum intensity threshold)
- `intensity_scale = 0.001` (intensity precision scaling factor)

## Scaling Behavior

**Sequential-ASCII Encoding**: Message length scales linearly in the m/z dimension only. As the message grows longer, the maximum m/z value increases proportionally while the intensity dimension remains bounded by the ASCII character range (0-128).

**Hilbert Curve Encoding**: Message length increases in both m/z and intensity dimensions simultaneously. The 2D Hilbert curve projection causes longer messages to expand the coordinate space in both axes, creating a more compact but two-dimensional growth pattern.

In [ ]:
order = 10
c = 4**10 / 128
c

In [ ]:
text = ['a']*3000
message = ''.join(text)
encoder = SpectraCodec()
SpectraCodec = reload(spectra_codec).SpectraCodec
encoder.convert_text_to_vector(message)
encoder.determine_hilbert_curve_order(len(message))
coords = np.column_stack((encoder.x_indices, encoder.y_indices))
indices = np.argwhere(np.asarray(encoder.encoded_ascii_vector)!=0).flatten()
# # print(coords[indices,:])
mz, intensity = encoder.scale_coords(coords, indices)
mz
# v,d = encoder.make_spectra_to_coords(mz,intensity)

In [ ]:
message = 'cat'
from importlib import reload
import spectra_codec
SpectraCodec = reload(spectra_codec).SpectraCodec
for my_order in range(7, 13):
    encoder = SpectraCodec(order=my_order) 
    encoder.convert_text_to_vector(message)
    # encoder.convert_text_to_vector('abc')
    coords = np.column_stack((encoder.x_indices, encoder.y_indices))
    indices = np.argwhere(np.asarray(encoder.encoded_ascii_vector)!=0).flatten()
    # # print(coords[indices,:])
    mz, intensity = encoder.scale_coords(coords, indices)
    v,d = encoder.make_spectra_to_coords(mz,intensity)
    # sort numpy arrays
    # idx = np.argsort(mz)
    # mz = mz[idx]
    # intensity = intensity[idx]
    # print(my_order)
    # print(mz)
    # print(intensity)
    # print()

In [ ]:
from importlib import reload
import figures_for_manuscript
reload(figures_for_manuscript)
figures_for_manuscript.make_comparison_figure()
figures_for_manuscript.make_abcd_figure()

In [ ]:
encoder = SpectraCodec(order=4)
print(len(encoder.x_indices),len(list(set(encoder.x_indices))))

In [ ]:
from SpectraCodec import SpectraCodec
# initialize the encoder with order = 2
encoder = SpectraCodec(order=4)
# encoder = SpectraCodec
# encoder.order = 2
# encoder.__init__()
message = ['ca']
message = ''.join(message)
z = encoder.convert_text_to_vector(message)
if z is None:
    print("Encoding failed.")
else:
    # encoder.convert_text_to_vector('abc')
    coords = np.column_stack((encoder.x_indices, encoder.y_indices))
    indices = np.argwhere(np.asarray(encoder.encoded_ascii_vector)!=0).flatten()
    # print(coords[indices,:])
    mz, intensity = encoder.scale_coords(coords, indices)


In [ ]:
with open('candide.txt','r') as f:
    message = f.read()

encoder = SpectraCodec()


# Encode the message
mz,intensity = encoder.sequential_encode_message(message)

# Plot the encoded message
fig,ax = plt.subplots()
ax.vlines(mz, 0, intensity, color='blue', lw=2)
ax.set_xlabel('m/z')
ax.set_ylabel('Intensity')
ax.set_title('Sequential-ASCII Encoding of Message')

In [ ]:
with open('candide.txt','r') as f:
    message = f.read()
encoder.convert_text_to_vector(message)
# encoder.convert_text_to_vector('abc')
coords = np.column_stack((encoder.x_indices, encoder.y_indices))
indices = np.argwhere(np.asarray(encoder.encoded_ascii_vector)!=0).flatten()
# # print(coords[indices,:])
mz, intensity = encoder.scale_coords(coords, indices)
intensity = intensity - encoder.min_intensity
fig,ax = plt.subplots()
# ax.plot(encoder.x_indices[indices],encoder.y_indices[indices],'.')
ax.vlines(mz,0,intensity)
ax.set_xlabel('m/z')
ax.set_ylabel('Intensity')
ax.set_title('Hilbert Curve Encoding of Message')
plt.show()
# len(encoder.encoded_ascii_vector) / 4 / 128

In [ ]:
encoder.x_indices.shape[0]/128

In [ ]:
from importlib import reload
import SpectraCodec
reload(SpectraCodec)
encoder = SpectraCodec.SpectraCodec()

In [ ]:
# message = "the quick brown fox jumped over the lazy dog"
# use a linear ascii to get the byte location of each character
original_filename = 'your_file.mzML'
output_filename = 'output.mzML'

encoder.encode_message_to_file(message, original_filename, output_filename, method='sequential')

In [ ]:
output_filename = 'output.mzML'
message= encoder.decode_message_from_file(output_filename, method='sequential')
print(message[:1000])


In [ ]:
from SpectraCodec import SpectraCodec
encoder = SpectraCodec()
# message = "the quick brown fox jumped over the lazy dog"

my_filename = '20180918_KBL_TM_Lakes_GEODES_All3_QE-HF_HILICZ-VF1_USHXG01161_NEG_MSMS_17_GEO-SP-24-UF_1_Rg70to1050-CE102040-0-4-S1_Run72.mzML'
filename_dict = encoder.filename_to_dict(my_filename.replace('.mzML', '')) 
# # # get the print representation of the dictionary as a string
message = str(filename_dict)

original_filename = 'your_file.mzML'
output_filename = 'output.mzML'

encoder.encode_message_to_file(message, original_filename, output_filename)

In [ ]:
import os
import time
from datetime import datetime
def get_acqtime_from_mzml(mzml_file):
    startTimeStamp=None
    with open(mzml_file) as mzml:
        for line in mzml:
            if 'startTimeStamp' in line:
                startTimeStamp = line.split('startTimeStamp="')[1].split('"')[0].replace('T',' ').rstrip('Z')
                break
#     print startTimeStamp
    if not '-infinity' in startTimeStamp:
        date_object = datetime.strptime(startTimeStamp, '%Y-%m-%d %H:%M:%S')
        utc_timestamp = int(time.mktime(date_object.timetuple()))
    else:
        utc_timestamp = int(0)
    return utc_timestamp

my_dir = '.'
filename = 'your_file.mzML'
filename = os.path.join(my_dir, filename)
# get the acquisition time from the mzML file
acquisition_time = get_acqtime_from_mzml(filename)
print(f"Acquisition time: {acquisition_time}")

In [ ]:
encoder = SpectraCodec()
output_filename = 'output.mzML'
message = encoder.decode_message_from_file(output_filename)
print(message)

In [ ]:
encoder = SpectraCodec()
import numpy as np
import matplotlib.pyplot as plt
# encoder.order = 12
# encoder.generate_full_hilbert_curve()
# encoder.x_indices[:10]
original_text = "the quick brown fox jumped over the lazy dog"
encoder.convert_text_to_vector(original_text)
# encoder.convert_text_to_vector('abc')
coords = np.column_stack((encoder.x_indices, encoder.y_indices))
indices = np.argwhere(np.asarray(encoder.encoded_ascii_vector)!=0).flatten()
# print(coords[indices,:])
mz_benglish, intensity_benglish = encoder.scale_coords(coords, indices)

fig,ax = plt.subplots()
# ax.plot(encoder.x_indices[indices],encoder.y_indices[indices],'.')
ax.vlines(mz_benglish,0,intensity_benglish)
ax.set_xlabel('m/z')
ax.set_ylabel('Intensity')
plt.show()
# len(encoder.encoded_ascii_vector) / 4 / 128

In [ ]:

# add spike signals to the m/z and intensity values
num_new_peaks = 100
mz = np.random.uniform(mz_benglish.min(), mz_benglish.max(), num_new_peaks)
intensity = np.random.uniform(intensity_benglish.min(), intensity_benglish.max(), num_new_peaks)
mz_benglish = np.concatenate([mz_benglish, mz])
intensity_benglish = np.concatenate([intensity_benglish, intensity])
idx = np.argsort(mz_benglish)
mz_benglish = mz_benglish[idx]
intensity_benglish = intensity_benglish[idx]


In [ ]:
mz_intensity_coords,peak_diffs = encoder.make_spectra_to_coords(mz_benglish, intensity_benglish)


In [ ]:
peak_diffs = np.abs(peak_diffs)
idx = np.argwhere(peak_diffs.max(axis=1)<0.001).flatten()
decoded_text = encoder.convert_coords_to_chars(mz_intensity_coords[idx,:])
# compare decoded_text to original_text
errors = 0
for i in range(len(original_text)):
    if original_text[i] != decoded_text[i]:
        errors +=1
print(f"Errors: {errors}")
print(f"Original text: {original_text}")
print(f"Decoded text: {decoded_text}")

In [ ]:
encoder = SpectraCodec()
coords = encoder.generate_full_hilbert_curve(order=12)
coords.shape[1]/128/1800


In [ ]:
encoder = SpectraCodec()
coords = encoder.generate_full_hilbert_curve(6)
x = coords[0,:]
y = coords[1,:]

fig,ax = plt.subplots(figsize=(6,6))
# color the line based on the index
for i in range(len(x)-1):
    ax.plot(x[i:i+2],y[i:i+2],color=plt.cm.cool(i/len(x)),linewidth=2)
#
print(len(x))
ax.set_facecolor('black')
ax.set_xlabel('m/z coordinate',fontsize=20)
ax.set_ylabel('Intensity coordinate',fontsize=20)

# increase the font size of the ticks
ax.tick_params(axis='both', which='major', labelsize=18)

In [ ]:
encoder = SpectraCodec()

my_filename = '20180918_KBL_TM_Lakes_GEODES_All3_QE-HF_HILICZ-VF1_USHXG01161_NEG_MSMS_17_GEO-SP-24-UF_1_Rg70to1050-CE102040-0-4-S1_Run72.mzML'
filename_dict = encoder.filename_to_dict(my_filename.replace('.mzML', '')) 
# # # get the print representation of the dictionary as a string
my_text = str(filename_dict)



# my_text = """Be not afeard; the isle is full of noises,
# Sounds and sweet airs, that give delight, and hurt not.
# Sometimes a thousand twangling instruments
# Will hum about mine ears; and sometime voices,
# That, if I then had waked after long sleep,
# Will make me sleep again: and then, in dreaming,
# The clouds methought would open, and show riches
# Ready to drop upon me; that, when I waked,
# I cried to dream again."""


# Convert text to vector
encoder.convert_text_to_vector(my_text)
coords = np.column_stack((encoder.x_indices, encoder.y_indices))
indices = np.argwhere(np.asarray(encoder.encoded_ascii_vector)!=0).flatten()
# print(coords[indices,:])
mz_benglish, intensity_benglish = encoder.scale_coords(coords, indices)
# Generate coordinates from the text vector
# coords = np.array([encoder.hilbert_index_to_xy(i) for i in range(len(text_vector))])


In [ ]:


# fig,ax = plt.subplots(1,1,figsize=(5,5))
# ax.scatter(coords[:,0], coords[:,1], c=text_vector, cmap='jet',s=text_vector*20)   
# # ax.axis('equal')
# ax.set_xlabel('mz index',fontsize=20)
# ax.set_ylabel('intensity index',fontsize=20)
# # increase the tick label
# ax.tick_params(axis='both', which='major', labelsize=20)
# # make the x and y axis not be scientific notation
# ax.ticklabel_format(useOffset=False, style='plain')
# # rotate x
# plt.xticks(rotation=45, ha='right')
# plt.show()


In [ ]:
# fig,ax = plt.subplots()
# ax.vlines(mz,0,intensity)
# ax.set_xlabel('mz')
# ax.set_ylabel('intensity')
# # turn off scientific notation
# ax.ticklabel_format(useOffset=False, style='plain')
# ax.set_ylim(min_intensity*0.95, 1.05*max(intensity))
# plt.show()


In [ ]:
# Example usage:
# coords is an array of shape (N,2). Each row is (x,y).
# Reconstruct indices and then build your 1D vector in the original ordering:
# idx = np.argwhere(text_vector == 1).flatten()
# mz_benglish, intensity_benglish = encoder.scale_coords(coords, idx)
mz_intensity_coords,diffs = encoder.make_spectra_to_coords(mz_benglish, intensity_benglish)

decoded_text = encoder.convert_coords_to_chars(mz_intensity_coords)
print(decoded_text)


In [ ]:
fig,ax = plt.subplots(1,1,figsize=(10,3))
ax.vlines(mz_benglish,0,intensity_benglish)

In [ ]:
# # now try to read the first spectrum with pyteomics instead of pymzml

# with mzml.read(my_mzML) as reader:
#     for spectrum in reader:
#         mz = spectrum['m/z array']
#         intensity = spectrum['intensity array']
#         break
    

In [ ]:

#         # idx = np.argwhere((self.x_indices == x) & (self.y_indices == y)).flatten()
# encoder = benglish()
# mz_intensity_coords,diffs = encoder.make_spectra_to_coords(mz, intensity)

# # # diffs = np.abs(diffs)
# # idx = np.argwhere((mz_intensity_coords[:,0] >= 0) & (mz_intensity_coords[:,1] >= 0) & (mz_intensity_coords[:,0] <= 4096) & (mz_intensity_coords[:,1] <4096)).flatten()



# # # idx2 = encoder.y_indices==mz_intensity_coords[:,1]
# # # idx = np.argwhere(idx1).flatten()
# decoded_text = encoder.convert_coords_to_chars(mz_intensity_coords)
# print(decoded_text)
# # mz_intensity_coords

In [ ]:
# import pandas as pd
# df = pd.DataFrame({'mz':mz[idx], 'intensity':intensity[idx],'diff_mz':diffs[idx,0],'diff_intensity':diffs[idx,1],'mz_coords':mz_intensity_coords[idx,0],'intensity_coords':mz_intensity_coords[idx,1]})
# df = df[df['mz_coords'] >= 0]
# df = df[df['intensity_coords'] >= 0]
# df['diff_mz'] = df['diff_mz'].abs()
# df['diff_intensity'] = df['diff_intensity'].abs()
# df['max_diff'] = df[['diff_mz','diff_intensity']].max(axis=1)
# df.sort_values('max_diff',ascending=False).head(10)

In [ ]:
# new_mz_values = [5.0001,4.9999,0.0001,10.0007,99.9999, 100.0001,100.0005, 800.0005,800.0001,799.9999,100.0001]  # Example m/z values

# def encode_array(array, format_char):
#     return base64.b64encode(struct.pack('<' + format_char * len(array), *array)).decode('ascii')
# mz_format = 'f' # d for 64 bit and f for 32 bit
# new_mz_encoded = encode_array(new_mz_values, mz_format)
# print(new_mz_encoded)

# # convert new_mz_encoded back into a list of numbers
# if mz_format=='f':
#     mz_decoded = struct.unpack('<' + 'f' * (len(new_mz_encoded) * 3 // 4 // 4), base64.b64decode(new_mz_encoded))
# else:
#     mz_decoded = struct.unpack('<' + 'd' * (len(new_mz_encoded) * 3 // 4 // 8), base64.b64decode(new_mz_encoded))
# mz_decoded = np.asarray(mz_decoded).round(4)
# print(mz_decoded - np.array(new_mz_values))

In [ ]:
# class ASCIIBinaryCodec:
#     """
#     Encodes and decodes text using standard 7-bit ASCII binary representation.
    
#     Each ASCII character (0-127) is represented by a unique 7-element array
#     of binary digits (1s and 0s).
#     """
    
#     def encode_char(self, char):
#         """
#         Convert a single character to its 7-element binary representation.
        
#         Example:
#             'A' -> [1, 0, 0, 0, 0, 0, 1]  (ASCII 65)
#             'a' -> [1, 1, 0, 0, 0, 0, 1]  (ASCII 97)
#             '!' -> [0, 1, 0, 0, 0, 0, 1]  (ASCII 33)
#         """
#         if not isinstance(char, str) or len(char) != 1:
#             raise ValueError("Input must be a single character")
        
#         # Get ASCII value of the character
#         ascii_val = ord(char)
        
#         # Check if it's within ASCII range
#         if ascii_val > 127:
#             raise ValueError(f"Character '{char}' is outside standard ASCII range (0-127)")
        
#         # Convert to 7-bit binary and return as a list of digits
#         binary = format(ascii_val, '07b')
#         return [int(bit) for bit in binary]
    
#     def encode(self, text):
#         """
#         Convert a string to a list of 7-element binary arrays.
        
#         Example:
#             "Hi" -> [[0, 1, 0, 0, 1, 0, 0], [1, 1, 0, 1, 0, 0, 1]]
#         """
#         return [self.encode_char(char) for char in text]
    
#     def decode_binary(self, binary):
#         """
#         Convert a 7-element binary array to a character.
        
#         Example:
#             [1, 0, 0, 0, 0, 0, 1] -> 'A'
#             [1, 1, 0, 0, 0, 0, 1] -> 'a'
#         """
#         if not isinstance(binary, list) or len(binary) != 7:
#             raise ValueError("Input must be a 7-element list of binary digits (0s and 1s)")
        
#         for bit in binary:
#             if bit not in (0, 1):
#                 raise ValueError("Binary array must contain only 0s and 1s")
                
#         # Convert binary list to binary string, then to decimal
#         binary_str = ''.join(str(bit) for bit in binary)
#         ascii_val = int(binary_str, 2)
        
#         # Convert the ASCII value to a character
#         return chr(ascii_val)
    
#     def decode(self, binary_arrays):
#         """
#         Convert a list of 7-element binary arrays to a string.
        
#         Example:
#             [[0, 1, 0, 0, 1, 0, 0], [1, 1, 0, 1, 0, 0, 1]] -> "Hi"
#         """
#         return ''.join(self.decode_binary(binary) for binary in binary_arrays)
    


# # Create an instance of the codec
# codec = ASCIIBinaryCodec()

# # Encode a message
# text = "Hello, World!"
# encoded = codec.encode(text)
# print(encoded)
# # Output: [[0, 1, 0, 0, 1, 0, 0], [1, 1, 0, 0, 1, 0, 1], ...] 

# # Decode the message
# decoded = codec.decode(encoded)
# print(decoded)  
# # Output: "Hello, World!"

In [ ]:
def select_k_important_features(X,attribute_values,k=1000):
    from sklearn.feature_selection import f_classif, SelectKBest

    # Note: You might need to transpose your data since sklearn expects
    # (n_samples, n_features) but your matrix is (n_features, n_samples)
    X_transposed = X.T
    F_values, p_values = f_classif(X_transposed, attribute_values)

    # Or to select top k features:
    selector = SelectKBest(f_classif, k=k)
    X_selected = selector.fit_transform(X_transposed, attribute_values)
    # Get the indices of the selected features
    selected_indices = selector.get_support(indices=True)
    # convert to a boolean mask for rows
    selected_mask = selector.get_support()
    # convert to a boolean mask for columns

    return selected_mask

min_intensity = 1e6
# filter feature_df to remove rows that are not changing for particular attributes
important_attributes = ['experimental_variable_A']#,'experimental_variable_B']
important_features = []
for attribute in important_attributes:
    # get the unique values for the attribute
    my_files = pd.notna(metadata_df.loc[attribute])
    # get the files that have that attribute
    my_files = metadata_df.columns[my_files]
    my_attributes = metadata_df.loc[attribute,my_files].to_list()
    vals = feature_df[my_files].values
    my_feature_index = feature_df.index.values
    # look for rows of vals that have some trend with my_attributes
    max_of_row = vals.max(axis=1)
    min_of_row = vals.min(axis=1)
    # elminate low intensity features
    idx = np.argwhere(max_of_row > min_intensity).flatten()
    my_feature_index = my_feature_index[idx]
    vals = vals[idx,:]
    max_of_row = vals.max(axis=1)
    min_of_row = vals.min(axis=1)
    # normalize vals so each row is between 0 and 1
    vals = (vals - min_of_row[:,None]) / (max_of_row[:,None] - min_of_row[:,None])
    idx = select_k_important_features(vals,my_attributes,k=1000)
    my_feature_index = my_feature_index[idx]
    important_features.append(my_feature_index)

important_features = np.unique(np.concatenate(important_features))
len(important_features)
# filter feature_df to only keep important features
feature_df = feature_df.loc[important_features,:]

In [ ]:
# make a tsne plot of the feature_df
from sklearn.manifold import TSNE

X = feature_df.values
X = X.T
idx1 = pd.notna(metadata_df.loc['experimental_variable_A'])
idx2 = ~metadata_df.loc['experimental_variable_A'].str.contains('NA', na=False)
idx = idx1 & idx2
X = X[idx,:]
X = X.astype(np.float32)
tsne = TSNE(n_components=2, random_state=0)
X_tsne = tsne.fit_transform(X)
fig,ax = plt.subplots(1,1,figsize=(8,6))
var_A = metadata_df.loc['experimental_variable_A',metadata_df.columns[idx]].to_list()
var_A = [v.split(':')[1].strip() for v in var_A]
var_A = ['%s days'%v for v in var_A]
# Create the scatter plot with categorical colors
# Get unique categories
unique_categories = sorted(list(set(var_A)))

# Create a scatter plot for each category
for category in unique_categories:
    indices = [i for i, x in enumerate(var_A) if x == category]
    ax.scatter(X_tsne[indices, 0], X_tsne[indices, 1], 
               label=category, marker='o', s=100, alpha=1)

# Add the legend
ax.legend(title='Incubation Time', loc='upper right', fontsize=14,title_fontsize=16)
ax.set_xlim(-60, 60)
ax.set_ylim(-60, 60)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('t-SNE 1', fontsize=16)
ax.set_ylabel('t-SNE 2', fontsize=16)
# increase the tick label
ax.tick_params(axis='both', which='major', labelsize=16)
# remove the upper and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

In [ ]:
my_text = """{
  "sample": "soil",
  "chromatography": "C18",
  "exp_var": "moisture:35%",
  "time": "day:42",
  "MS": "Q-Exactive",
  "ionization": "ESI+"
}"""

encoder = SpectraCodec()
encoder.convert_text_to_vector(my_text)

# get the one-hot encoding of the text
v = encoder.encoded_ascii_vector
v = v.reshape(-1,128)
fig,ax = plt.subplots(1,1,figsize=(4,4))
ax.imshow(v, aspect='auto', cmap='hot')
ax.set_xlabel('ASCII Character', fontsize=16)
ax.set_ylabel('Character in String', fontsize=16)

coords = np.column_stack((encoder.x_indices, encoder.y_indices))
indices = np.argwhere(np.asarray(encoder.encoded_ascii_vector)!=0).flatten()
mz_benglish, intensity_benglish = encoder.scale_coords(coords, indices)
fig,ax = plt.subplots(1,1,figsize=(9,2.8))
ax.vlines(mz_benglish,0,intensity_benglish)
ax.set_xlabel('m/z',fontsize=16)
ax.set_ylabel('Intensity',fontsize=16)
ax.set_ylim(intensity_benglish.min(), intensity_benglish.max()*1.001)

In [ ]:
my_files = metadata_df.columns.tolist()
my_text = metadata_df[my_files[0]].to_dict().__str__()
encoder = SpectraCodec()
encoder.convert_text_to_vector(my_text)
coords = np.column_stack((encoder.x_indices, encoder.y_indices))
indices = np.argwhere(np.asarray(encoder.encoded_ascii_vector)!=0).flatten()
mz_benglish, intensity_benglish = encoder.scale_coords(coords, indices)
fig,ax = plt.subplots(1,1,figsize=(15,3))
ax.vlines(mz_benglish,0,intensity_benglish)
ax.set_xlabel('m/z',fontsize=16)
ax.set_ylabel('Intensity',fontsize=16)
ax.set_ylim(intensity_benglish.min(), intensity_benglish.max()*1.001)